# Tutorial 1
This tutorial covers, how to generate tight binding files for a homoatomic system (graphene) using the Slater-Koster rules

## 1. Generating the basis functions

Numerical atomic orbitals (NAO) serve as basis functions, generated with an atomic DFT calculation. Spherical confinement potentials can be used to tune the basis functions to match properties like a band structure. 

Import modules to handle confinement, atomic dft, H/S Slater-Koster tables and dipole Slater-Koster tables

In [14]:
from hotcent.confinement import PowerConfinement
from hotcent.atomic_dft import AtomicDFT
from hotcent.pos_op.offsite_twocenter_new import Offsite2cTable
from hotcent.pos_op.offsite_twocenter_posop import Offsite2cTablePosOp


Set atom type, DFT functional and confinement parameters:

In [15]:

element = 'C'
xc = 'GGA_X_PBE+GGA_C_PBE'
conf = PowerConfinement(r0=50.0, s=4)
r0 = 3.2 # Bohr
wf_conf = {'2s': PowerConfinement(r0=r0, s=8.2),
           '2p': PowerConfinement(r0=r0, s=8.2),
           }


Set up atomic DFT and run, save eigenvalues of weakly confined atom (for DFT convergence) and run again with stronger confinement:

In [16]:

atom = AtomicDFT(element,
                xc = xc,
                 confinement=conf,
                 perturbative_confinement=False,
                 configuration='[He] 2s2 2p2',
                 valence=['2s', '2p'],
                 scalarrel=True,
                 maxiter=2500,
                 timing=False,
                 nodegpts=150,
                 mix=0.2,
                 txt='-',
                 )
atom.run()
eigenvalues=atom.enl

atom.set_confinement(conf)
atom.set_wf_confinement(wf_confinement=wf_conf)
atom.run()

*********************************************************************
Scalar-relativistic all-electron GGA_X_PBE+GGA_C_PBE calculator for C
*********************************************************************
max 1 nodes, 300 grid points
Applying PowerConfinement(r0=50.000000, s=4.000000, ubound=1.0e+04)
to get the confined electron density
and the confined 2s and 2p subshell(s)

Start iteration...
iter   0, dn=4.0e+00>1.0e-07, max 7 sp-iter
iter  10, dn=7.8e-02>1.0e-07, max 3 sp-iter
iter  20, dn=1.6e-03>1.0e-07, max 2 sp-iter
iter  30, dn=1.4e-04>1.0e-07, max 2 sp-iter
iter  40, dn=1.6e-05>1.0e-07, max 2 sp-iter
iter  50, dn=1.7e-06>1.0e-07, max 1 sp-iter
iter  60, dn=1.9e-07>1.0e-07, max 1 sp-iter

valence subshell eigenvalues:
-----------------------------
  2s:   -0.506080361334
  2p:   -0.194786159233

energy contributions:
----------------------------------------
sum of eigenvalues:     -21.496150484312
Hartree energy:         17.725334593833
vxc correction:         -6.51343659


## 2. Generating the Slater-Koster files. 

The NAO and the respective eigenvalues from the atomic DFT calculation are used to generate Slater-Koster parameter files. These files allow the calculation of overlap, Hamiltonian and dipole elements between basis functions centered at arbitrary positions.


Create the Slater-Koster files for H and S (output naming scheme E1-E2.skf)

In [17]:

rmin, dr, N = 0.4, 0.02, 900
off2c = Offsite2cTable(atom, atom, timing=True)
off2c.run(rmin, dr, N, xc=xc, nr=200, ntheta=400, wflimit=1e-9)
off2c.write(format="unique", eigenvalues=eigenvalues)  # writes to default C-C.skf

Wave function range for C = 5.66925 a0
R=    0.40, 73464 grid points ...
R=    2.20, 100208 grid points ...
R=    4.00, 107394 grid points ...
R=    5.80, 62134 grid points ...
R=    7.60, 19068 grid points ...
R=    9.40, 6226 grid points ...
R=   11.20, 192 grid points ...
Writing to C-C.skf



Create the Slater-Koster files for position operator (output naming scheme E1-E2_dipole.skf)

In [18]:

rmin, dr, N = 0.4, 0.02, 900
off2c = Offsite2cTablePosOp(atom, atom, timing=False)
off2c.run(rmin, dr, N, nr=200, ntheta=400, wflimit=1e-9)
off2c.write_dipole(format='unique')


Timing:
         label                       time     calls    %sub  %covered   %tot
-------------------------------------------------------------------------------
MultiAtomIntegrator                31.066         1 (100.0 %, 99.9 %) 100.0 % |000000000000000000000000000000|
../run_offsite2c                   31.024         1 ( 99.9 %, 99.0 %)  99.9 % |111111111111111111111111111111|
../../make_grid                    13.803       547 ( 44.5 %        )  44.4 % |2222222222222|
../../calculate_offsite2c          16.907       546 ( 54.5 %, 62.7 %)  54.4 % |2222222222222222|
../../../prelude                    5.624       546 ( 33.3 %, 96.9 %)  18.1 % |33333|
../../../../veff                    5.452       546 ( 96.9 %        )  17.5 % |44444|
../../../vsigma                     4.972      3276 ( 29.4 %        )  16.0 % |33333|
 -------------------------------------------------------------------------------
total time       31.066 seconds      0 h  0 min  31 sec
Thu Sep  3 09:54:31 2026
W

## 3. Generating tigh-binding parameters
Based on the Slater-Koster parameter files, the matrix elements for overlap, Hamiltonian and dipole elements can be calculated for arbitrary molecular geometries. All interactions within the cutoff of the Slater-Koster file are considered. The output can be generated in the tight-binding format used by Wannier90.

In order to do that, create a directory for the SK files and another one for the dipole SK files. And remove the _dipole suffix. Additionally a geometry file for the system of interest that can be read by `ase` is required. 

In [19]:
from ase.build import graphene
from hotcent.pos_op.mat_elem_evaluation import SlaterKosterIntegrator

In [20]:
!mkdir skfiles
!mkdir skfiles_pos_op
!mv C-C.skf skfiles
!mv C-C_dipole.skf skfiles_pos_op/C-C.skf

mkdir: cannot create directory ‘skfiles’: File exists


mkdir: cannot create directory ‘skfiles_pos_op’: File exists



Specify the maximal angular momentum quantum number to include for each element. The integrator takes in an `ase.Atoms` object and the directory paths for the H/S and the position operator parameter files. `write_seedname()` saves as the tight binding format used in Wannier90:

In [21]:

max_l = {'C':1, 'H':0, 'S':2, 'Mo':2}
graphene = graphene('CC', size=(1,1,1), vacuum=10)
lcao_graphene = SlaterKosterIntegrator(graphene, skpath="skfiles", maxl_dict=max_l, skpath_posop="skfiles_pos_op")
lcao_graphene.write_seedname()